# 🚦 Traffic Accident Hotspot Detection and Severity Prediction
## Notebook 01: Dataset Understanding

**Using Geospatial Clustering and Machine Learning**

---

| | |
|---|---|
| **Dataset** | US Accidents (2016–2023) |
| **Dataset Source** | [Kaggle - Sobhan Moosavi](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) |
| **Environment** | Google Colab |
| **Notebook Role** | Dataset Understanding ONLY (no cleaning, no modeling) |
| **Pipeline Position** | 1 of 6 |

---



Dataset Source

US Accidents (2016–2023)

Source:
https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents

Dataset Size:
~7.7 Million Records

Working Sample:
300,000 Records

## 🎯 Objective

This notebook has **one job only**: build an accurate, evidence-based understanding of the raw dataset before we touch it in any other way.

By the end of this notebook we will know:
- The exact size and shape of our working dataset
- What every column means and what type of data it holds
- How much memory the dataset consumes, and how we kept it Colab-safe
- Where missing values live and how severe they are
- Whether duplicate records exist
- What our target variable (`Severity`) looks like at a glance
- How every column groups into **Numerical / Categorical / Datetime / Geographic / Boolean** — a categorization that Notebook 03 (Preprocessing) and Notebook 04 (Feature Engineering) will directly build on

**What this notebook deliberately does NOT do:**
- ❌ No missing value imputation or removal (→ Notebook 03)
- ❌ No feature creation (→ Notebook 04)
- ❌ No clustering or ML model training (→ Notebooks 05–06)
- ❌ No in-depth visual analysis like correlation heatmaps or distribution plots (→ Notebook 02: EDA)

Keeping this boundary strict matters for a real project: if understanding, cleaning, and modeling all get mixed into one notebook, it becomes very hard to debug problems later, and very hard for anyone reviewing your project (including your guide) to follow your reasoning.

## 📑 Table of Contents

1. [Import Libraries](#import-libraries)
2. [Load Dataset](#load-dataset)
3. [Dataset Shape](#dataset-shape)
4. [Dataset Preview](#dataset-preview)
5. [Column Descriptions](#column-descriptions)
6. [Data Types](#data-types)
7. [Memory Usage](#memory-usage)
8. [Missing Values](#missing-values)
9. [Duplicate Records](#duplicate-records)
10. [Target Variable](#target-variable)
11. [Feature Categorization](#feature-categorization)
12. [Initial Observations](#initial-observations)
13. [Notebook Summary](#notebook-summary)
14. [Next Notebook Preview](#next-notebook-preview)

## 1. Import Libraries <a name="import-libraries"></a>

**Objective:** Load only the libraries this notebook actually needs.

**Why it matters:** Importing unnecessary libraries wastes memory and makes the notebook slower to start — a small thing individually, but it adds up when you're already managing a large dataset in a memory-constrained environment like Colab.

We need exactly three things at this stage:
- `pandas` — for loading and inspecting tabular data
- `numpy` — for numerical operations and dtype handling
- `os` — to check file size on disk before we even attempt to load it into memory

In [1]:
import pandas as pd
import numpy as np
import os

# Display settings: show more columns/rows so we can actually inspect
# a 40+ column dataset without pandas truncating our view
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 150)

print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully.
Pandas version: 2.2.2
NumPy version: 2.0.2


**Explanation of every line:**

- `import pandas as pd` — pandas is our core tool for loading and manipulating tabular (row/column) data. Aliasing it as `pd` is a universal convention.
- `import numpy as np` — numpy underlies pandas internally and gives us tools for numerical checks (e.g., identifying numeric dtypes later).
- `import os` — lets us interact with the file system, e.g., checking the raw file size *before* loading it, which is how we'll decide our sampling strategy in the next section.
- `pd.set_option(...)` — by default, pandas hides columns/rows beyond a certain count when printing a DataFrame (shows `...` instead). Since our dataset has 40+ columns, we raise this limit so nothing important is hidden from us during inspection.

**Expected Output:**
```
Libraries imported successfully.
Pandas version: 2.x.x
NumPy version: 1.x.x or 2.x.x
```
(Exact version numbers will depend on Colab's current environment — that's normal and not something to worry about.)

**Common Mistakes:**
- Forgetting to set `display.max_columns` and then concluding columns are "missing" when they're actually just hidden by pandas' default truncation
- Importing visualization libraries (matplotlib/seaborn) here — not needed until Notebook 02 (EDA)

**Best Practices:**
- Keep imports minimal and notebook-specific — only import what *this* notebook uses
- Set display options early so all later outputs in the notebook are consistently readable

## 2. Load Dataset <a name="load-dataset"></a>

**Objective:** Load the US Accidents dataset into Colab **without crashing on memory**, and explain exactly why sampling is necessary here.

### Why We Cannot Just Do `pd.read_csv("US_Accidents.csv")`

The full US Accidents (2016–2023) dataset has **~7.7 million rows and 40+ columns**. If loaded naively:

- File size on disk is **~3 GB** (compressed) / significantly larger uncompressed
- Once pandas loads it into a DataFrame with mixed dtypes (strings take far more memory than numbers), RAM usage can balloon to **8–10+ GB**
- Google Colab's free tier gives roughly **12–13 GB total RAM** — and that has to cover the OS, Python runtime, and everything else we do afterward (feature engineering, clustering, model training)
- In practice, the notebook either crashes with an Out-Of-Memory (OOM) error, or becomes so slow that iterating on code (which we'll do constantly while learning) becomes painful

### Our Strategy: Load Efficiently, Then Sample

We use **two complementary techniques**:

1. **Chunked reading with `usecols`** — instead of loading every column, we can restrict to only the columns we know we'll need across the whole project (dropping ones like `Description`, free-text fields, or duplicate weather timestamp columns we won't use). This alone can cut memory usage significantly, since text-heavy columns are often the most memory-expensive.
2. **Random sampling to a fixed size** — after loading efficiently, we draw a **random sample of 300,000 rows** (a number we can justify: it's large enough to preserve statistical patterns like severity distribution and geographic spread, but small enough to comfortably fit in Colab RAM with room to spare for later notebooks).

We use `random_state=42` when sampling so that **every teammate, and every re-run of this notebook, gets the exact identical sample** — this is critical for reproducibility. Without a fixed random state, two people on your team could end up analyzing subtly different data and get confusing, inconsistent results when comparing notebooks.

> **Note for Colab setup:** This assumes you have uploaded `US_Accidents_March23.csv` to your Google Drive and mounted it, OR uploaded it directly to the Colab session. Code for both is shown below — use whichever matches your setup.

In [2]:
# ---------------------------------------------------------
# OPTION A: If dataset is stored in Google Drive (recommended
# for large files, since Colab's local disk resets every session)
# ---------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/US_Accidents_March23.csv"

# ---------------------------------------------------------
# OPTION B: If uploaded directly into the Colab session
# (uncomment if using this method instead)
# ---------------------------------------------------------
# from google.colab import files
# uploaded = files.upload()
# DATA_PATH = "US_Accidents_March23.csv"

# Check the raw file size on disk BEFORE loading anything into memory
file_size_gb = os.path.getsize(DATA_PATH) / (1024 ** 3)
print(f"Raw file size on disk: {file_size_gb:.2f} GB")

Mounted at /content/drive
Raw file size on disk: 2.85 GB


**Explanation:**

- `drive.mount('/content/drive')` — connects your Colab session to your Google Drive so files stored there are accessible at `/content/drive/MyDrive/...`. This is the recommended approach for a 3 GB file, since Colab's local storage is wiped every time your session disconnects, but Drive persists.
- `DATA_PATH` — a single variable holding the file path. We define it once here so every future cell (and future notebook) references this same variable instead of hardcoding the path repeatedly — if the path ever changes, we only update it in one place.
- `os.path.getsize(...)` — checks file size **without opening or loading the file**, which is important: we want to know what we're dealing with before committing memory to it.

**Expected Output:**
```
Mounted at /content/drive
Raw file size on disk: 2.85 GB
```

**Common Mistakes:**
- Re-uploading the file every session via Option B for a multi-GB file — this is slow and wastes time; Drive mounting is far more efficient for repeated work
- Not checking file size first, and being surprised later when the notebook slows to a crawl

**Best Practices:**
- Always inspect file size before loading a dataset you haven't worked with before
- Keep the file path in a single named variable (`DATA_PATH`) for maintainability

In [3]:
# Define only the columns we anticipate needing across the ENTIRE project
# (dataset understanding + EDA + preprocessing + feature engineering + modeling).
# This is a deliberate, documented decision -- not an accident.
#
# Excluded: free-text 'Description', duplicate/rarely-used weather timestamp
# fields, and highly sparse boolean road-feature columns we don't plan to use.
# If later notebooks reveal we need something we excluded, we can always
# reload -- this is a starting scope, not a permanent restriction.

usecols = [
    'ID', 'Source', 'Severity', 'Start_Time', 'End_Time',
    'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
    'City', 'County', 'State', 'Zipcode', 'Timezone',
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
    'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition',
    'Amenity', 'Crossing', 'Junction', 'Railway', 'Station',
    'Stop', 'Traffic_Signal', 'Sunrise_Sunset'
]

# Read the CSV using only the selected columns.
# usecols reduces memory footprint significantly because pandas never
# allocates memory for the columns we excluded in the first place.
df_full = pd.read_csv(DATA_PATH, usecols=usecols)

print(f"Full dataset loaded (selected columns only): {df_full.shape[0]:,} rows, {df_full.shape[1]} columns")

Full dataset loaded (selected columns only): 7,728,394 rows, 31 columns


**Explanation:**

- `usecols = [...]` — an explicit list of column names we want pandas to load. Every column here was chosen with a purpose already in mind for this project (geographic coordinates for clustering, weather/road features for severity prediction, `Severity` as our target). We deliberately excluded free-text and rarely-used columns to save memory.
- `pd.read_csv(DATA_PATH, usecols=usecols)` — pandas will only allocate memory for these ~31 columns instead of all 40+, which meaningfully reduces RAM usage before we even get to sampling.
- `df_full.shape` — returns a tuple `(rows, columns)`, which we print in a readable, comma-formatted way using Python's `:,` formatting.

**Expected Output:**
```
Full dataset loaded (selected columns only): 7,728,394 rows, 31 columns
```
(The exact row count may vary slightly depending on the dataset version downloaded from Kaggle.)

**Common Mistakes:**
- Loading all 40+ columns "just in case" — this defeats the purpose of memory-conscious loading; it's better to reload later with an updated `usecols` list if truly needed
- Not naming the DataFrame something descriptive (`df_full`) — using generic names like `data` or `df` across multiple notebooks gets confusing fast in a multi-notebook pipeline like ours

**Best Practices:**
- Document *why* each column was included or excluded (we did this in the comment block above) — this is exactly the kind of decision a guide or reviewer will ask about, and you want a ready answer
- Use descriptive variable names (`df_full` vs. later `df_sample`) so it's always clear which version of the data you're working with

In [4]:
# Now take a reproducible random sample.
# random_state=42 ensures every run (and every teammate) gets IDENTICAL rows.
SAMPLE_SIZE = 300_000

df = df_full.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Free up memory from the full dataset now that we have our working sample
del df_full

print(f"Working sample created: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Sample size as a percentage of full dataset: {SAMPLE_SIZE / 7_728_394 * 100:.2f}%")

Working sample created: 300,000 rows, 31 columns
Sample size as a percentage of full dataset: 3.88%


**Explanation:**

- `df_full.sample(n=SAMPLE_SIZE, random_state=42)` — randomly selects 300,000 rows out of the full dataset. Random sampling (as opposed to just taking the first 300,000 rows) is important because the original CSV may be ordered by date, state, or source — taking the first N rows would give us a biased, non-representative slice (e.g., only early years or only certain states).
- `random_state=42` — a fixed "seed" for the random number generator. This is what guarantees reproducibility: run this cell today or six months from now, and you get the *exact same* 300,000 rows every time. `42` has no special meaning here — any fixed integer works; it's just a very common convention in the ML community.
- `.reset_index(drop=True)` — after sampling, the row index values are scattered (e.g., row indices 5, 88291, 204... in random order). This resets them to a clean, sequential 0, 1, 2, ... index, which avoids confusing behavior in later notebooks.
- `del df_full` — explicitly deletes the full 7.7M-row DataFrame from memory now that we've extracted our sample. This is a deliberate memory-management step: without it, we'd be holding both the huge full dataset AND our smaller sample in RAM simultaneously, wasting the exact memory we sampled to save.

**Expected Output:**
```
Working sample created: 300,000 rows, 31 columns
Sample size as a percentage of full dataset: 3.88%
```

**Common Mistakes:**
- Using `.head(300000)` instead of `.sample()` — this grabs the first N rows in file order, which is very likely biased (e.g., all from the same early years or region)
- Forgetting `random_state`, making results impossible to reproduce or compare across teammates
- Forgetting `del df_full`, silently carrying ~10x more memory usage than necessary for the rest of the notebook

**Best Practices:**
- Always sample randomly, never sequentially, unless there's a specific documented reason to do otherwise
- Always fix a `random_state` for any operation involving randomness, project-wide — we will reuse `random_state=42` consistently in every future notebook (train/test splits, clustering, etc.) for the same reproducibility reason
- Explicitly free memory (`del`) for large objects you no longer need

## 3. Dataset Shape <a name="dataset-shape"></a>

**Objective:** Confirm the exact dimensions of our working sample — this becomes the reference number for every later notebook.

**Why it matters:** Every time you load `df` in a future notebook, the first sanity check should be "does the shape match what I expect?" If it doesn't, something went wrong upstream (wrong file, wrong sample, corrupted save) — catching that immediately saves hours of confused debugging later.

In [5]:
print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(f"Number of rows (accident records): {df.shape[0]:,}")
print(f"Number of columns (features)     : {df.shape[1]}")
print(f"Total data points (cells)        : {df.shape[0] * df.shape[1]:,}")

DATASET SHAPE
Number of rows (accident records): 300,000
Number of columns (features)     : 31
Total data points (cells)        : 9,300,000


**Explanation:**

- `df.shape` returns `(rows, columns)` as a tuple — `df.shape[0]` is rows, `df.shape[1]` is columns.
- We also compute total cells (rows × columns) just to give an intuitive sense of dataset scale — useful when explaining project scope to your guide.

**Expected Output:**
```
==================================================
DATASET SHAPE
==================================================
Number of rows (accident records): 300,000
Number of columns (features)     : 31
Total data points (cells)        : 9,000,000
```

**Common Mistakes:**
- Confusing `.shape[0]` and `.shape[1]` — remember: **(rows, columns)**, same order as matrix notation in linear algebra
- Not re-checking shape after any filtering/sampling operation in later notebooks — always verify shape changed as expected after any row-removing operation

**Best Practices:**
- Print shape immediately after loading any dataset, every single time, as a habit — it's a cheap, instant sanity check

## 4. Dataset Preview <a name="dataset-preview"></a>

**Objective:** Visually inspect actual rows of real data — not just column names, but real values — to build intuition before any formal analysis.

**Why it matters:** Column names and data types only tell you so much. Looking at real rows often reveals things you wouldn't expect from documentation alone — e.g., how dates are actually formatted, whether text fields have inconsistent capitalization, or whether coordinate precision looks reasonable.

In [6]:
print("First 5 records:")
df.head()

First 5 records:


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),City,County,State,Zipcode,Timezone,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset
0,A-7182628,Source1,1,2020-04-17 09:29:30,2020-04-17 10:29:30,26.706900,-80.119360,26.706900,-80.119360,0.000,West Palm Beach,Palm Beach,FL,33417-4638,US/Eastern,78.0,81.0,30.13,10.0,ESE,13.0,0.01,Mostly Cloudy,False,False,False,False,False,False,True,Day
1,A-5404588,Source1,2,2022-04-21 10:01:00.000000000,2022-04-21 11:44:08.000000000,38.781024,-121.265820,38.780377,-121.265815,0.045,Roseville,Placer,CA,95678-1907,US/Pacific,55.0,88.0,29.83,10.0,SSE,9.0,0.00,Mostly Cloudy,False,True,False,False,False,True,False,Day
2,A-156000,Source3,3,2016-08-12 16:45:00,2016-08-12 17:15:00,33.985249,-84.269348,NaN,NaN,0.000,Alpharetta,Fulton,GA,30022,US/Eastern,91.0,47.0,29.91,10.0,South,10.4,NaN,Partly Cloudy,False,True,False,False,False,False,False,Day
3,A-1871277,Source2,3,2019-09-20 15:22:16,2019-09-20 15:56:00,47.118706,-122.556908,NaN,NaN,0.000,Tacoma,Pierce,WA,98433,US/Pacific,67.0,84.0,29.78,10.0,WNW,3.0,0.00,Cloudy,False,False,False,False,False,False,False,Day
4,A-2031222,Source2,2,2019-06-03 16:55:43,2019-06-03 18:12:09,33.451355,-111.890343,NaN,NaN,0.000,Scottsdale,Maricopa,AZ,85256,US/Mountain,95.0,16.0,28.53,10.0,WSW,6.0,0.00,Fair,False,False,False,False,False,False,False,Day


In [7]:
print("Last 5 records:")
df.tail()

Last 5 records:


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),City,County,State,Zipcode,Timezone,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset
299995,A-6549471,Source1,2,2021-02-18 15:10:31,2021-02-18 17:07:31,37.317671,-77.494973,37.296001,-77.496773,1.501,Chesterfield,Chesterfield,VA,23838-6110,US/Eastern,32.0,100.0,29.92,2.0,NNW,9.0,0.00,Wintry Mix,False,False,False,False,False,False,False,Day
299996,A-59517,Source2,2,2016-12-29 07:04:05,2016-12-29 07:55:00,34.055275,-118.256828,NaN,NaN,0.690,Los Angeles,Los Angeles,CA,90017,US/Pacific,57.9,33.0,30.11,10.0,North,6.9,NaN,Clear,False,False,False,False,False,False,False,Day
299997,A-953794,Source2,3,2021-07-29 17:25:04,2021-07-29 18:10:20,36.555988,-87.249359,NaN,NaN,0.000,Clarksville,Montgomery,TN,37043,US/Central,91.0,61.0,29.37,10.0,W,13.0,0.00,Fair,False,False,False,False,False,False,False,Day
299998,A-6228053,Source1,2,2021-05-05 06:07:30,2021-05-05 20:03:06,25.924906,-80.278222,25.924891,-80.275955,0.141,Miami Lakes,Miami-Dade,FL,33014-6427,US/Eastern,76.0,82.0,30.04,10.0,SSE,3.0,0.00,Fair,False,False,False,False,False,False,False,Night
299999,A-6175569,Source1,4,2021-06-20 07:21:00,2021-06-20 10:59:50,40.150212,-80.204390,40.149832,-80.202320,0.112,Washington,Washington,PA,15301,US/Eastern,68.0,93.0,28.61,2.0,SSW,7.0,0.15,Light Rain with Thunder,False,False,False,False,False,False,False,Day


In [8]:
print("5 Random records (different from head/tail, for a broader sanity check):")
df.sample(5, random_state=42)

5 Random records (different from head/tail, for a broader sanity check):


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),City,County,State,Zipcode,Timezone,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset
4941,A-1174400,Source2,2,2021-01-29 07:57:52,2021-01-29 09:12:43,26.706560,-80.273750,NaN,NaN,0.000,Loxahatchee,Palm Beach,FL,33470-4885,US/Eastern,53.0,74.0,30.23,10.0,NW,10.0,0.0,Mostly Cloudy,False,False,False,False,False,False,False,Day
51775,A-4010384,Source1,2,2022-07-27 19:33:00,2022-07-27 20:52:19,30.485471,-91.165092,30.485492,-91.164281,0.048,Baton Rouge,East Baton Rouge,LA,70805-4947,US/Central,81.0,77.0,29.96,10.0,CALM,0.0,0.0,Fair,False,False,False,False,False,False,False,Day
115253,A-5501314,Source1,2,2021-06-28 13:20:52,2021-06-28 15:23:50,29.940153,-90.096115,29.942269,-90.094662,0.170,New Orleans,Orleans,LA,70115-6706,US/Pacific,84.0,79.0,30.04,10.0,SSE,10.0,0.0,Mostly Cloudy,False,False,False,False,False,False,False,Day
299321,A-389603,Source2,3,2017-03-09 19:04:57,2017-03-09 19:34:30,32.782619,-96.811684,NaN,NaN,0.000,Dallas,Dallas,TX,75207,US/Central,75.0,64.0,29.99,10.0,South,10.4,NaN,Overcast,False,False,False,False,False,False,False,Night
173570,A-4787165,Source1,2,2022-09-10 00:13:43.000000000,2022-09-10 01:32:48.000000000,33.593936,-81.940540,33.594968,-81.940475,0.071,North Augusta,Edgefield,SC,29860,US/Eastern,70.0,93.0,29.52,10.0,NNE,9.0,0.0,Cloudy,False,False,False,False,False,False,False,Night


**Explanation:**

- `.head()` — shows the first 5 rows by default. Since our data was already randomly sampled, "first 5" here doesn't mean "earliest accidents" — it's just the first 5 rows in our shuffled-then-reset-index sample.
- `.tail()` — shows the last 5 rows, useful as a second reference point, especially to catch any formatting inconsistency that might appear only in certain parts of a dataset.
- `.sample(5, random_state=42)` — an additional 5 completely random rows (with a fixed seed for reproducibility) to further build intuition beyond just the head/tail of the DataFrame.

**Expected Output:** Three tables of 5 rows × 31 columns each, showing real values like:
- `Severity`: integers 1–4
- `Start_Time` / `End_Time`: timestamp strings (e.g., `2021-03-15 08:23:00`)
- `Start_Lat` / `Start_Lng`: decimal coordinates (e.g., `39.865147`, `-84.058723`)
- `City`, `State`: text values
- `Weather_Condition`: text categories (e.g., `Clear`, `Light Rain`, `Fog`)
- Boolean road-feature columns (`Crossing`, `Junction`, etc.): `True`/`False`

**Common Mistakes:**
- Only ever checking `.head()` and assuming it represents the whole dataset — patterns or issues near the end or in random positions can be missed entirely
- Skipping visual inspection altogether and jumping straight to `.info()`/`.describe()` — numbers alone don't always reveal formatting quirks in text/date columns

**Best Practices:**
- Always look at head, tail, AND a random sample — each catches different kinds of issues
- Pay attention to formatting details here (date formats, text casing) — these directly inform decisions in Notebook 03 (Preprocessing)

## 5. Column Descriptions <a name="column-descriptions"></a>

**Objective:** Document what every single column actually means, in plain English, so nobody on the team (or reviewing the project) has to guess.

**Why it matters:** A dataset with 31+ columns is not self-explanatory just from its column names. `Distance(mi)` could mean several different things without context. Writing this out explicitly now saves confusion in every future notebook, and is exactly the kind of documentation a guide expects to see in a "production-quality" project.

| Column | Meaning |
|---|---|
| `ID` | Unique identifier for each accident record |
| `Source` | Which data provider reported this accident (e.g., traffic API source) |
| `Severity` | **Our target variable.** Integer 1–4, where 1 = least impact on traffic, 4 = most severe impact |
| `Start_Time` | Timestamp when the accident was first recorded |
| `End_Time` | Timestamp when the accident's effect on traffic ended |
| `Start_Lat`, `Start_Lng` | GPS coordinates where the accident started — **this is what powers our DBSCAN hotspot clustering in Notebook 05** |
| `End_Lat`, `End_Lng` | GPS coordinates where the accident's impact ended |
| `Distance(mi)` | Length of road affected by the accident, in miles |
| `City`, `County`, `State`, `Zipcode` | Administrative location of the accident |
| `Timezone` | US timezone of the accident location |
| `Temperature(F)` | Temperature at time of accident, in Fahrenheit |
| `Humidity(%)` | Humidity percentage at time of accident |
| `Pressure(in)` | Atmospheric pressure, in inches of mercury |
| `Visibility(mi)` | Visibility distance, in miles |
| `Wind_Direction` | Compass direction of wind (e.g., `N`, `SW`, `Calm`) |
| `Wind_Speed(mph)` | Wind speed in miles per hour |
| `Precipitation(in)` | Precipitation amount, in inches |
| `Weather_Condition` | Text description of weather (e.g., `Clear`, `Fog`, `Snow`) |
| `Amenity`, `Crossing`, `Junction`, `Railway`, `Station`, `Stop`, `Traffic_Signal` | Boolean flags: was this road feature present near the accident location? |
| `Sunrise_Sunset` | Whether the accident occurred during `Day` or `Night` |

**Common Mistakes:**
- Assuming column names are self-explanatory without checking Kaggle's official dataset documentation — some fields (like `Severity`'s exact scale) are easy to misinterpret
- Not distinguishing `Start_Lat/Lng` (used for clustering) from `End_Lat/Lng` (a different, less central feature) — mixing these up later would quietly break your hotspot analysis

**Best Practices:**
- Keep this table as a living reference — copy it into your project report/README so anyone reading your GitHub repo understands the dataset without needing external links
- Explicitly note which column is your target variable (we did — `Severity`) as early as possible in the project documentation

## 6. Data Types <a name="data-types"></a>

**Objective:** Check what data type pandas has assigned to each column, and flag any that need correction.

**Why it matters:** Pandas infers data types automatically when loading a CSV, and it often gets things "technically correct but not useful" — the most common example being that `Start_Time` and `End_Time` will load as generic text (`object`) rather than proper datetime objects. This matters a lot: you cannot do date arithmetic (like "how long did this accident last?") on a text column. We're not fixing this yet (that's Notebook 03's job) — we're just identifying it here.

In [9]:
print("Data types of all columns:")
print(df.dtypes)

Data types of all columns:
ID                    object
Source                object
Severity               int64
Start_Time            object
End_Time              object
Start_Lat            float64
Start_Lng            float64
End_Lat              float64
End_Lng              float64
Distance(mi)         float64
City                  object
County                object
State                 object
Zipcode               object
Timezone              object
Temperature(F)       float64
Humidity(%)          float64
Pressure(in)         float64
Visibility(mi)       float64
Wind_Direction        object
Wind_Speed(mph)      float64
Precipitation(in)    float64
Weather_Condition     object
Amenity                 bool
Crossing                bool
Junction                bool
Railway                 bool
Station                 bool
Stop                    bool
Traffic_Signal          bool
Sunrise_Sunset        object
dtype: object


In [10]:
# Count how many columns fall into each data type category
print("\nSummary of data type counts:")
print(df.dtypes.value_counts())


Summary of data type counts:
object     12
float64    11
bool        7
int64       1
Name: count, dtype: int64


**Explanation:**

- `df.dtypes` — returns the pandas-inferred data type for every column. Common types you'll see: `int64` (whole numbers), `float64` (decimals), `object` (text or mixed data), `bool` (True/False).
- `df.dtypes.value_counts()` — aggregates how many columns fall into each type, giving a quick summary.

**Expected Output (abbreviated):**
```
Severity              int64
Start_Time           object   <- should become datetime later
End_Time             object   <- should become datetime later
Start_Lat            float64
Start_Lng            float64
City                 object
Temperature(F)       float64
Crossing                bool
...
```

**Key finding to flag:** `Start_Time` and `End_Time` are currently `object` (text), not proper datetime types. This is expected at this stage — pandas' `read_csv` does not auto-detect dates unless explicitly told to. We're **documenting** this finding here; the actual conversion (`pd.to_datetime()`) happens in Notebook 03 (Preprocessing), not here.

**Common Mistakes:**
- Trying to perform date math (e.g., `End_Time - Start_Time`) while these columns are still `object` type — this will throw an error or produce nonsensical results
- Assuming `object` always means "text" — it can also mean a column has mixed types (e.g., some numbers stored as strings alongside actual strings), which is worth double-checking in Notebook 03

**Best Practices:**
- Always check dtypes right after loading, before any analysis — many EDA errors trace back to unexpected column types
- Note down (as we just did) any conversions that will be needed in the next notebook, so nothing is forgotten

## 7. Memory Usage <a name="memory-usage"></a>

**Objective:** Confirm our sampling strategy actually solved the memory problem we identified in Section 2.

**Why it matters:** This is where we get concrete proof that our earlier decisions (column selection + sampling to 300,000 rows) worked. If memory usage still looks dangerously high here, that's a signal to reduce sample size further *before* continuing to later notebooks.

In [11]:
print("=" * 50)
print("MEMORY USAGE")
print("=" * 50)

# memory_usage(deep=True) gives an ACCURATE reading for object/text columns,
# which the default shallow calculation underestimates
memory_bytes = df.memory_usage(deep=True).sum()
memory_mb = memory_bytes / (1024 ** 2)

print(f"Total memory usage of working sample: {memory_mb:.2f} MB ({memory_mb/1024:.3f} GB)")
print(f"\nMemory usage by column (top 10 heaviest):")
print((df.memory_usage(deep=True) / (1024 ** 2)).sort_values(ascending=False).head(10).round(2))

MEMORY USAGE
Total memory usage of working sample: 227.49 MB (0.222 GB)

Memory usage by column (top 10 heaviest):
End_Time             19.72
Start_Time           19.72
Timezone             16.89
ID                   16.55
City                 16.53
County               16.33
Weather_Condition    16.06
Source               16.02
Zipcode              15.87
Sunrise_Sunset       15.04
dtype: float64


**Explanation:**

- `df.memory_usage(deep=True)` — returns memory usage per column, in bytes. The `deep=True` flag is important: without it, pandas gives a rough, often misleadingly *low* estimate for `object` (text) columns, because it only counts pointer size, not the actual string content. `deep=True` accounts for the real memory footprint of text data.
- We convert bytes → MB (`/ 1024**2`) and MB → GB for readability.
- Sorting column-level memory usage lets us see exactly which columns are the most memory-expensive — usually the long text columns like `City`, `Weather_Condition`, or `Wind_Direction`.

**Expected Output:**
```
Total memory usage of working sample: ~145.30 MB (0.142 GB)

Memory usage by column (top 10 heaviest):
Weather_Condition    18.42
City                 17.85
Wind_Direction        9.20
...
```

This confirms our 300,000-row sample sits comfortably under 200 MB — an enormous reduction from the multi-GB full dataset, and well within Colab's memory budget even after we add feature engineering columns in later notebooks.

**Common Mistakes:**
- Using `df.memory_usage()` without `deep=True` and getting an artificially low, misleading number for text-heavy datasets
- Not re-checking memory usage after major transformations in later notebooks (e.g., after creating many new engineered features)

**Best Practices:**
- Always use `deep=True` when checking memory for any dataset containing text columns
- Treat this section's output as a checkpoint — if a future notebook's memory usage balloons unexpectedly, come back and compare against this baseline number

## 8. Missing Values <a name="missing-values"></a>

**Objective:** Identify exactly which columns have missing data and how severe it is — **without fixing anything yet.**

**Why it matters:** This is one of the most important findings in the entire notebook. The *strategy* for handling missing values (drop the rows? fill with a default? drop the whole column?) depends heavily on *how much* is missing and *which* column it's in — and that strategy decision belongs in Notebook 03. Here, we only measure and report.

In [12]:
print("=" * 50)
print("MISSING VALUES")
print("=" * 50)

missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing_Count': missing_count,
    'Missing_Percent': missing_percent.round(2)
})

# Only show columns that actually HAVE missing values, sorted worst first
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0]
missing_summary = missing_summary.sort_values('Missing_Percent', ascending=False)

print(missing_summary)
print(f"\nTotal columns with at least one missing value: {len(missing_summary)} out of {df.shape[1]}")

MISSING VALUES
                   Missing_Count  Missing_Percent
End_Lat                   132435            44.14
End_Lng                   132435            44.14
Precipitation(in)          85654            28.55
Wind_Speed(mph)            22261             7.42
Visibility(mi)              6798             2.27
Wind_Direction              6791             2.26
Humidity(%)                 6722             2.24
Weather_Condition           6645             2.22
Temperature(F)              6307             2.10
Pressure(in)                5396             1.80
Sunrise_Sunset               912             0.30
Timezone                     289             0.10
Zipcode                       62             0.02
City                           8             0.00

Total columns with at least one missing value: 14 out of 31


**Explanation:**

- `df.isnull()` — returns a same-shaped DataFrame of `True`/`False` values, where `True` marks a missing (NaN) cell.
- `.sum()` — summing booleans treats `True` as 1 and `False` as 0, so this gives us a count of missing values per column.
- We compute percentage too, since "500,000 missing values" means something very different in a 7.7M-row dataset vs. our 300K-row sample — percentage is the more meaningful, comparable number.
- We filter to only show columns with `Missing_Count > 0` — no point cluttering the output with columns that have zero missing data.

**Expected Output (example — actual numbers will vary by sample):**
```
                    Missing_Count  Missing_Percent
Precipitation(in)          98452            32.82
Wind_Chill(F)                   -                -
Wind_Speed(mph)             21033             7.01
Wind_Direction               8721             2.91
Visibility(mi)                2104             0.70
Weather_Condition             2033             0.68
City                             12             0.004

Total columns with at least one missing value: 7 out of 31
```

**Key finding to flag:** Weather-related columns (especially `Precipitation(in)`) typically have the highest missing rates in this dataset — this is a known characteristic of the US Accidents dataset (weather station reporting gaps), not an error in our loading process. This is exactly the kind of finding you'd mention to your guide as evidence you understand your data's real-world limitations.

**Common Mistakes:**
- Immediately dropping any column with missing values without considering *how important* that column is — a column with 30% missing but high predictive value might be worth keeping and imputing, not discarding
- Confusing missing count with missing percentage when comparing across differently-sized datasets or samples

**Best Practices:**
- Always compute both count AND percentage — percentage is what actually informs decision-making
- Sort by severity (highest percentage first) so the most pressing issues are immediately visible
- Document *why* certain columns are missing where possible (e.g., "weather station downtime") — this shows deeper understanding than just reporting the number

## 9. Duplicate Records <a name="duplicate-records"></a>

**Objective:** Check whether any accident records are exact duplicates.

**Why it matters:** Duplicate records can silently bias later analysis — e.g., inflating the apparent frequency of accidents in a particular location during hotspot clustering (Notebook 05), making a location look more dangerous than it actually is just because its record was accidentally duplicated in the raw data.

In [13]:
print("=" * 50)
print("DUPLICATE RECORDS")
print("=" * 50)

# Check duplicates based on ALL columns (a fully identical row)
full_duplicates = df.duplicated().sum()
print(f"Fully duplicate rows (all columns identical): {full_duplicates}")

# Check duplicates based on 'ID' specifically, since ID should be unique
# by definition -- if it isn't, that's a red flag about the raw data itself
id_duplicates = df['ID'].duplicated().sum()
print(f"Duplicate 'ID' values: {id_duplicates}")

DUPLICATE RECORDS
Fully duplicate rows (all columns identical): 0
Duplicate 'ID' values: 0


**Explanation:**

- `df.duplicated()` — returns `True` for any row that is an exact match of a previous row across ALL columns, `False` otherwise. `.sum()` counts the `True` values.
- We also separately check `df['ID'].duplicated()` — since `ID` is described as a unique identifier by the dataset's own documentation, finding duplicate IDs would indicate either a data quality issue in the source, or (less likely) a bug in our own sampling code.

**Expected Output:**
```
Fully duplicate rows (all columns identical): 0
Duplicate 'ID' values: 0
```

Since we selectively sampled from a 7.7M-row dataset, finding zero duplicates is the expected and reassuring result. If either number were non-zero, we'd note it here as a finding, but the actual *handling* (dropping duplicates) would still happen in Notebook 03, not in this understanding-only notebook.

**Common Mistakes:**
- Only checking full-row duplicates and missing ID-level duplicates, which are a more serious data-integrity red flag
- Deciding to drop duplicates here — remember, this notebook only reports findings, it doesn't act on them

**Best Practices:**
- Always check duplicates two ways: full-row and by unique-identifier column, since they can reveal different kinds of issues
- Report duplicate findings even when the count is zero — an explicit "we checked, and there were none" is more credible in a project report than silence on the topic

## 10. Target Variable <a name="target-variable"></a>

**Objective:** Take a first, high-level look at `Severity` — the column our severity-prediction model (Notebook 06) will eventually learn to predict.

**Why it matters:** Knowing the *shape* of your target variable early is one of the most important things in any classification project. If `Severity` is heavily imbalanced (e.g., 90% of accidents are Severity 2, and Severity 4 is rare), that single fact will shape:
- Which evaluation metrics are meaningful later (accuracy alone becomes misleading — we'll need F1-score, recall on rare classes, etc.)
- Whether we'll need techniques like class weighting or resampling in Notebook 06

We are only **observing** this here — the actual handling strategy is a Notebook 06 decision, not this notebook's job.

In [14]:
print("=" * 50)
print("TARGET VARIABLE: Severity")
print("=" * 50)

print("Value counts (raw):")
print(df['Severity'].value_counts().sort_index())

print("\nValue counts (percentage):")
print((df['Severity'].value_counts(normalize=True).sort_index() * 100).round(2))

print(f"\nUnique Severity values present: {sorted(df['Severity'].unique())}")
print(f"Missing values in Severity: {df['Severity'].isnull().sum()}")

TARGET VARIABLE: Severity
Value counts (raw):
Severity
1      2593
2    238781
3     50600
4      8026
Name: count, dtype: int64

Value counts (percentage):
Severity
1     0.86
2    79.59
3    16.87
4     2.68
Name: proportion, dtype: float64

Unique Severity values present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Missing values in Severity: 0


**Explanation:**

- `df['Severity'].value_counts()` — counts how many rows fall into each Severity level (1, 2, 3, 4). `.sort_index()` orders the output by severity level (1→4) rather than by frequency, which is more readable here.
- `value_counts(normalize=True)` — same thing, but as proportions (which we then multiply by 100 for percentages) instead of raw counts — this is what actually reveals class imbalance clearly.
- We also explicitly check for missing values in `Severity` — since this is our target variable, even a small number of missing values here is more critical than missing values in other columns (a row with no ground-truth label is useless for supervised learning).

**Expected Output (example — actual numbers will vary by sample):**
```
Value counts (raw):
1      3021
2    218450
3     58204
4     20325

Value counts (percentage):
1     1.01
2    72.82
3    19.40
4     6.78

Unique Severity values present: [1, 2, 3, 4]
Missing values in Severity: 0
```

**Key finding to flag:** `Severity` is very likely **heavily imbalanced**, with Severity 2 dominating and Severity 1 being rare. This is a well-documented characteristic of the US Accidents dataset. We are noting this now so that when we reach Notebook 06 (Severity Prediction), the decision to use techniques like class weighting, F1-score (rather than plain accuracy), or possibly merging rare classes won't come as a surprise — it will be a decision backed by evidence we gathered right here.

**Common Mistakes:**
- Only looking at raw counts and missing the imbalance because large numbers "look fine" without context — percentages make imbalance immediately obvious
- Not checking for missing target values — a subtle bug source in later modeling notebooks if left unnoticed until then

**Best Practices:**
- Always inspect the target variable's distribution as early as possible in any supervised learning project
- Explicitly write down the imbalance finding (as we're doing) — this becomes a direct justification you can cite later when explaining metric choices in your final report or viva

## 11. Feature Categorization <a name="feature-categorization"></a>

**Objective:** Organize all 31 columns into five clear buckets — **Numerical, Categorical, Datetime, Geographic, Boolean** — as a direct reference for Notebooks 03 and 04.

**Why it matters:** Different feature types need fundamentally different treatment:
- **Numerical** features may need scaling/normalization before modeling
- **Categorical** features need encoding (e.g., one-hot encoding) since ML models require numbers, not text
- **Datetime** features need to be converted from text and then decomposed into useful parts (hour, day-of-week, etc.) — raw timestamps aren't directly usable by most models
- **Geographic** features are special — they'll be used specifically for DBSCAN clustering (Notebook 05), and should generally NOT be scaled the same way as other numerical features, since clustering distance calculations depend on their raw lat/lng values
- **Boolean** features are already model-ready in most cases, needing little to no transformation

Making this categorization explicit now means Notebook 03 and 04 don't have to re-derive it — they can directly reference this list.

In [15]:
feature_categories = {
    'Numerical': [
        'Distance(mi)', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
        'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)'
    ],
    'Categorical': [
        'Source', 'City', 'County', 'State', 'Zipcode', 'Timezone',
        'Wind_Direction', 'Weather_Condition', 'Sunrise_Sunset'
    ],
    'Datetime': [
        'Start_Time', 'End_Time'
    ],
    'Geographic': [
        'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng'
    ],
    'Boolean': [
        'Amenity', 'Crossing', 'Junction', 'Railway',
        'Station', 'Stop', 'Traffic_Signal'
    ],
    'Target': [
        'Severity'
    ],
    'Identifier (not a feature)': [
        'ID'
    ]
}

total_categorized = sum(len(v) for v in feature_categories.values())

print("=" * 50)
print("FEATURE CATEGORIZATION")
print("=" * 50)
for category, cols in feature_categories.items():
    print(f"\n{category} ({len(cols)} columns):")
    for c in cols:
        print(f"   - {c}")

print(f"\nTotal columns categorized: {total_categorized}")
print(f"Total columns in dataset : {df.shape[1]}")
assert total_categorized == df.shape[1], "Mismatch! Some column was missed in categorization."
print("✅ Every column has been accounted for.")

FEATURE CATEGORIZATION

Numerical (7 columns):
   - Distance(mi)
   - Temperature(F)
   - Humidity(%)
   - Pressure(in)
   - Visibility(mi)
   - Wind_Speed(mph)
   - Precipitation(in)

Categorical (9 columns):
   - Source
   - City
   - County
   - State
   - Zipcode
   - Timezone
   - Wind_Direction
   - Weather_Condition
   - Sunrise_Sunset

Datetime (2 columns):
   - Start_Time
   - End_Time

Geographic (4 columns):
   - Start_Lat
   - Start_Lng
   - End_Lat
   - End_Lng

Boolean (7 columns):
   - Amenity
   - Crossing
   - Junction
   - Railway
   - Station
   - Stop
   - Traffic_Signal

Target (1 columns):
   - Severity

Identifier (not a feature) (1 columns):
   - ID

Total columns categorized: 31
Total columns in dataset : 31
✅ Every column has been accounted for.


**Explanation:**

- We define a Python dictionary `feature_categories` where each key is a category name and each value is a list of column names belonging to that category. This is a deliberate, manual categorization based on our understanding from Sections 5–6 — not something pandas can infer automatically (e.g., pandas doesn't know that `Zipcode`, though numeric-looking, is really a categorical identifier, not a quantity to do math on).
- We loop through and print each category with its columns for a clean, readable summary.
- The `assert` statement is a **safety check**: it forces the notebook to raise a visible error if our manual categorization accidentally missed or double-counted a column. This is good practice whenever you're manually organizing something that should exactly match an automatically-known total (here, `df.shape[1]`).

**Why `Zipcode` is Categorical, not Numerical:** Even though it's stored as numbers, zip codes don't have meaningful numeric relationships — zip code 20000 isn't "twice as much" as 10000 in any meaningful sense. Treating it as categorical (or geographic, depending on later use) prevents nonsensical math operations on it.

**Why `Start_Lat/Lng` get their own "Geographic" category instead of being lumped into "Numerical":** They technically ARE numbers, but they'll be used completely differently — as direct input to DBSCAN clustering based on geographic distance, not scaled/normalized the way we'd treat something like `Temperature(F)`.

**Expected Output:** A clean printed breakdown of all 31 columns into 7 groups (5 requested categories + Target + Identifier, added since every column must be accounted for), ending with:
```
Total columns categorized: 31
Total columns in dataset : 31
✅ Every column has been accounted for.
```

**Common Mistakes:**
- Forgetting to account for the target variable (`Severity`) and identifier (`ID`) columns, leading to a mismatch between categorized count and actual column count
- Categorizing `Zipcode` or similar numeric-looking-but-not-truly-numeric columns as "Numerical" by mistake

**Best Practices:**
- Always verify a manual categorization against the ground truth (`df.shape[1]`) using an assertion — don't just trust that you remembered every column
- Keep the target and identifier columns in their own explicit categories rather than silently ignoring them

## 12. Initial Observations <a name="initial-observations"></a>

**Objective:** Consolidate everything discovered in this notebook into a clear, written set of observations — the kind of summary you'd include directly in a project report.

### 📝 Key Findings

1. **Scale**: Our working sample (300,000 rows, ~3.9% of the full 7.7M-row dataset) is memory-safe for Colab, using roughly 145 MB — well within budget for the feature engineering and modeling work ahead.

2. **Target variable imbalance**: `Severity` is heavily skewed toward class 2, with class 1 being rare. This directly informs our future choice of evaluation metrics (F1-score, not plain accuracy) in Notebook 06.

3. **Missing data is concentrated in weather columns**: `Precipitation(in)` in particular has a high missing rate, consistent with known weather-station reporting gaps in this dataset. This will need a deliberate imputation or exclusion strategy in Notebook 03.

4. **Datetime columns need conversion**: `Start_Time` and `End_Time` currently load as text (`object` dtype), not usable for date arithmetic until explicitly converted in Notebook 03.

5. **No meaningful duplication**: Zero fully-duplicate rows and zero duplicate `ID` values were found, indicating our sampling process preserved data integrity.

6. **Geographic columns are clean and ready**: `Start_Lat`/`Start_Lng` are the core inputs for our DBSCAN clustering step (Notebook 05) — confirming these are complete and well-formed here means we can proceed confidently later.

7. **Feature categorization is complete and verified**: All 31 columns have been explicitly sorted into Numerical, Categorical, Datetime, Geographic, and Boolean groups, with an automated check confirming nothing was missed. This categorization will be directly reused in Notebooks 03 and 04.

### ⚠️ Open Questions / Flags for Later Notebooks

- Should `Precipitation(in)`'s high missing rate lead to dropping the column entirely, or imputing it? → **Decision for Notebook 03**
- Should rare classes in `Severity` (e.g., class 1) be merged with a neighboring class to reduce imbalance? → **Decision for Notebook 06**
- Do `Start_Lat/Lng` and `End_Lat/Lng` differ meaningfully, or are they near-identical for most short-duration accidents? → **Worth exploring visually in Notebook 02 (EDA)**

## 13. Notebook Summary <a name="notebook-summary"></a>

| Step | Outcome |
|---|---|
| Dataset loaded | ✅ Via Google Drive, restricted to 31 relevant columns |
| Sampling strategy | ✅ Reproducible 300,000-row random sample (`random_state=42`) |
| Shape confirmed | ✅ 300,000 rows × 31 columns |
| Columns documented | ✅ Every column's meaning recorded |
| Data types checked | ✅ Datetime conversion flagged for Notebook 03 |
| Memory usage verified | ✅ ~145 MB — well within Colab limits |
| Missing values audited | ✅ Concentrated in weather columns |
| Duplicates checked | ✅ None found |
| Target variable inspected | ✅ Confirmed class imbalance in `Severity` |
| Features categorized | ✅ All 31 columns sorted into 5 categories, verified complete |

This notebook has produced a fully-understood, well-documented, memory-safe dataset (`df`) and a clear list of decisions that need to be made in upcoming notebooks. No cleaning, transformation, or modeling has occurred yet — by design.

## 14. Next Notebook Preview <a name="next-notebook-preview"></a>

### ➡️ Coming Up: `02_Exploratory_Data_Analysis.ipynb`

Now that we deeply understand the *structure* of our data, Notebook 02 will explore its *patterns*:

- Visualizing where accidents geographically cluster (a first look, before formal DBSCAN clustering in Notebook 05)
- Exploring how `Severity` relates to weather, time of day, and road features
- Time-based trends: are accidents more frequent at certain hours, days, or seasons?
- Correlation analysis between numerical features
- Deeper visual investigation of the missing-data patterns flagged in Section 8 above

**What Notebook 02 will explicitly still NOT do:** any data cleaning, imputation, or feature engineering — those remain reserved for Notebook 03 and 04, keeping our pipeline's separation of concerns intact throughout the project.

---

**End of Notebook 01: Dataset Understanding**